# Building Tools for Agents

*Notebook 04*

Give your agent real capabilities by turning Python functions into tools.

**Topics:**

- Turning any Python function into a tool with `@function_tool`

- Automatic schema generation — no manual JSON

- How the agent decides which tool to call

- Writing good tool descriptions — naming, inputs, outputs

---

## 🔧 Setup

In [ ]:
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"  # Course default — fast and cost-effective for most tasks

print("✅ Ready!")

---

## 🛠️ Part 1: Your First Tool

Without tools, an agent can only use knowledge baked into the model at training time. It has no access to your data — your databases, your files, your APIs. With tools, it can reach anything Python can reach.

### Before: Agent without tools

In [ ]:
message = "What department does employee E003 work in?"

agent_no_tools = Agent(
    name="NoToolsAgent",
    instructions="You are a helpful assistant.",
    model=MODEL
)

result = await Runner.run(agent_no_tools, input=message)

print(result.final_output)

### After: Agent with an employee lookup tool

In [ ]:
# Simulated employee database
EMPLOYEES = {
    "E001": {"name": "Sarah Chen", "department": "Engineering", "salary": 95000},
    "E002": {"name": "Marcus Johnson", "department": "Marketing", "salary": 78000},
    "E003": {"name": "Priya Patel", "department": "Finance", "salary": 88000},
    "E004": {"name": "Tom Rivera", "department": "Engineering", "salary": 102000},
    "E005": {"name": "Lisa Wong", "department": "HR", "salary": 72000},
}

@function_tool
def get_employee(employee_id: str) -> str:
    """Look up an employee's name, department, and salary by their employee ID."""
    employee = EMPLOYEES.get(employee_id.upper())
    if employee:
        return (
            f"Name: {employee['name']}, "
            f"Department: {employee['department']}, "
            f"Salary: ${employee['salary']:,}"
        )
    return f"No employee found with ID {employee_id}"

# --------------------------------------------------------------
print("✅ get_employee() ready")

The `@function_tool` decorator turns any Python function into an agent tool:

- **Name** — tells the agent what the tool is called
- **Docstring** — tells the agent what the tool does and when to use it
- **Type hints** — define the expected inputs; the parameter types generate the schema (the structured description the model reads to know what arguments the tool expects) automatically. No manual JSON required.

This is the core reason tools matter in real projects — the same pattern lets an agent answer questions from your live business data, internal APIs, or services instead of guessing from model knowledge.

#### Run Agent with Employee Lookup Tool

In [ ]:
agent_with_tools = Agent(
    name="EmployeeAgent",
    instructions="You are a helpful assistant.",
    model=MODEL,
    tools=[get_employee]
)

result = await Runner.run(agent_with_tools, input=message)

print(result.final_output)

Open the traces dashboard to confirm the agent selected `get_employee`, called it with the employee ID from your message, and used the result — this is what tool invocation looks like in practice.

---

## 🗓️ Part 2: A Tool with No Parameters

Not all tools take input. Some just return live information — the current time, system status, environment data.

In [ ]:
@function_tool
def get_current_datetime() -> str:
    """Return the current date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

# --------------------------------------------------------------
print("✅ get_current_datetime() ready")

#### Run Agent with DateTime Tool

In [ ]:
agent_datetime = Agent(
    name="DateTimeAgent",
    instructions="Use the get_current_datetime tool for all time and date questions.",
    model=MODEL,
    tools=[get_current_datetime]
)

result = await Runner.run(agent_datetime, input="What day of the week is it today?")

print(result.final_output)

---

## 🧠 Part 3: How the Agent Decides

When an agent has tools, it doesn't always use them. It decides based on the user's message and the tool's name and docstring — which means **your docstring is part of your prompt**. A clear, specific docstring helps the agent pick the right tool. A vague one leads to missed calls or wrong choices.

In [ ]:
# An agent with both tools — watch it choose
agent_multi = Agent(
    name="MultiToolAgent",
    instructions="You are a helpful assistant. Use tools whenever they are relevant.",
    model=MODEL,
    tools=[get_employee, get_current_datetime]
)

# --------------------------------------------------------------
print("✅ MultiToolAgent ready")

#### Test: Employee Question

In [ ]:
result = await Runner.run(agent_multi, input="What is employee E001's salary?")

print(result.final_output)

#### Test: Time Question

In [ ]:
result = await Runner.run(agent_multi, input="What time is it right now?")

print(result.final_output)

#### Test: No Tool Needed

### 💡 Writing a Good Tool Docstring

A docstring the agent can act on states three things:
1. **What it returns** — the specific data the tool gives back
2. **When to use it** — the trigger condition ("use this when the user asks about…")
3. **What each input means** — if it's not obvious from the parameter name

Example: `"Get employee info"` is too vague. A docstring that names the returned fields and states the use case works much better.

In [ ]:
result = await Runner.run(agent_multi, input="What is the capital of France?")

print(result.final_output)

---

## 💪 Practice Exercises

### Exercise 1: Word Counter Tool

*Covers: `@function_tool` — building and wiring custom tools*

Create a tool that counts the words in a string, then build an agent that uses it.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Word Counter Tool
# --------------------------------------------------------------
# Objective: Build a tool that counts words and an agent that uses it.

# TODO 1: Define a @function_tool called count_words
#             - Parameter: text (str)
#             - Returns: str with the word count
#             - Docstring: describe what it does clearly
#             - Hint: len(text.split()) gives you the word count

# TODO 2: Create an Agent with the count_words tool

# TODO 3: Run the agent with: "How many words are in this sentence: The quick brown fox"

# --- Write your code below this line ---

### Exercise 2: Temperature Converter

*Covers: `@function_tool` — tool inputs, outputs, and docstrings*

Create a tool that converts Celsius to Fahrenheit.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Temperature Converter
# --------------------------------------------------------------
# Objective: Build a tool that converts temperatures.

# TODO 1: Define a @function_tool called celsius_to_fahrenheit
#             - Parameter: celsius (float)
#             - Formula: (celsius * 9/5) + 32
#             - Returns: str with the result

# TODO 2: Create an Agent with the conversion tool

# TODO 3: Run the agent with: "What is 100 degrees Celsius in Fahrenheit?"

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**`@function_tool` turns any Python function into an agent capability:**

- The docstring tells the agent what the tool does and when to use it
- Type hints generate the parameter schema automatically — no manual JSON

**Tool selection is automatic:**

- The agent reads names, docstrings, and type hints to decide which tool fits the task
- No tool needed means no tool called

---

## 📍 Next Step

05: Writing Effective Agent Instructions — Learn to write system instructions that shape how your agent behaves, responds, and uses tools.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-04-building-tools-for-agents)

---